# Python Functions: Definitions, Arguments, and Return Values

Functions let you give a task a name, reuse its implementation, and make its inputs and results explicit. This lesson develops those ideas through greeting, calculation, and shopping-list examples.

## Learning objectives
By the end of this lesson, you should be able to:
- Define and call functions, and distinguish parameters from arguments.
- Match positional and keyword arguments to parameters.
- Return values and distinguish returned results from side effects.
- Explain local names, object mutation, and rebinding.
- Choose meaningful defaults and avoid accidentally shared mutable defaults.
- Use variable-length arguments and unpack collections into calls.
- Read positional-only and keyword-only function signatures.
- Document functions with docstrings and type annotations.
- Recognize generator functions and closures.

## Before you begin
Use Python 3.8 or later for the positional-only parameter examples.

## 1. Defining and calling functions

A function groups statements under a name. Calling the function executes its body. Functions help you:
- Hide implementation details behind a clear interface.
- Break a program into smaller tasks.
- Reuse code rather than copy and maintain repeated blocks.
- Test behavior through well-defined inputs and outputs.

The `def` statement contains a function name, parentheses, optional parameters, a colon, and an indented body. Function names normally use lowercase words separated by underscores.

In [ ]:
def greet(name):
    print(f"Hello, {name}!")

greet("Pythonista")

Hello, Pythonista!


### Definition is not execution

Executing `def` creates a function object and binds its name. The body runs when the function is called. Parentheses distinguish referring to a function from calling it.

In [ ]:
print("Is greet callable?", callable(greet))
another_greeting = greet
print("Same function object:", another_greeting is greet)
another_greeting("Pythonista")

Is greet callable? True
Same function object: True
Hello, Pythonista!


### Functions with no parameters

Empty parentheses mean the function has no parameters. They are still required in a call. This function reads a shopping list defined outside its body.

In [ ]:
shopping_list = {
    "Bread": 1,
    "Milk": 2,
    "Chocolate": 1,
    "Butter": 1,
    "Coffee": 1,
}

def show_global_list():
    for item_name, quantity in shopping_list.items():
        print(f"{quantity}x {item_name}")

show_global_list()

1x Bread
2x Milk
1x Chocolate
1x Butter
1x Coffee


This function depends on a global variable. Later, you will pass the dictionary explicitly so the display function can work with different shopping lists. A function’s inputs are not necessarily limited to its parameters if it also reads external state.

### Placeholders need a body

Use `pass` when a function is temporarily unfinished. An empty function body is not valid syntax. A stub containing `pass` runs without performing a task and returns `None`.

In [ ]:
def unfinished_task():
    pass

print(unfinished_task())

None


## 2. Parameters and argument binding

A *parameter* is a name in a function definition.

An *argument* is a value supplied in a call.

In `greet(name)`, `name` is the parameter; in `greet("Pythonista")`, the string is the argument.

### Positional arguments
Positional arguments bind to parameters from left to right.

The following function has three required parameters.

In [ ]:
def calculate_cost(item, quantity, price):
    print(f"{quantity} {item} cost ${quantity * price:.2f}")

calculate_cost("bananas", 6, 0.74)

6 bananas cost $4.44


### Order affects meaning

The first call below runs but reverses the intended quantity and unit price. The second gives the calculation incompatible types. A function call can be syntactically valid and still communicate the wrong meaning.

In [ ]:
calculate_cost("bananas", 0.74, 6)

try:
    calculate_cost(6, "bananas", 0.74)
except TypeError as error:
    print(type(error).__name__ + ":", error)

0.74 bananas cost $4.44
TypeError: can't multiply sequence by non-int of type 'float'


### Keyword arguments

Write `parameter=value` to name the parameter receiving a value. For parameters that accept keywords, this makes the call more explicit and allows the named arguments to appear in a different order. A required parameter is not necessarily positional-only.

In [ ]:
calculate_cost(item="bananas", quantity=6, price=0.74)
calculate_cost(price=0.74, quantity=6, item="bananas")
calculate_cost("bananas", quantity=6, price=0.74)

6 bananas cost $4.44
6 bananas cost $4.44
6 bananas cost $4.44


### Binding errors

A call must supply all required parameters without assigning one twice. Extra arguments are accepted only if the function signature supports them. Compare each message below with the function definition.

In [44]:
try:
    calculate_cost("bananas", 6)
except TypeError as error:
    print("Missing argument ->", type(error).__name__ + ":", error)

try:
    calculate_cost("bananas", 6, 0.74, "mango")
except TypeError as error:
    print("Extra argument ->", type(error).__name__ + ":", error)

try:
    calculate_cost(item="bananas", quantity=6, cost=0.74)
except TypeError as error:
    print("Unknown keyword ->", type(error).__name__ + ":", error)

try:
    calculate_cost("bananas", item="bananas", quantity=6, price=0.74)
except TypeError as error:
    print("Duplicate value ->", type(error).__name__ + ":", error)

try:
    calculate_cost(item="bananas", quantity=6)
except TypeError as error:
    print("Missing argument ->", type(error).__name__ + ":", error)

Missing argument -> TypeError: calculate_cost() missing 1 required positional argument: 'price'
Extra argument -> TypeError: calculate_cost() takes 3 positional arguments but 4 were given
Unknown keyword -> TypeError: calculate_cost() got an unexpected keyword argument 'cost'
Duplicate value -> TypeError: calculate_cost() got multiple values for argument 'item'
Missing argument -> TypeError: calculate_cost() missing 1 required positional argument: 'price'


### Ordinary positional arguments come before keywords

In an ordinary call that does not use unpacking, place positional arguments before keyword arguments.

The invalid call below is stored as text and compiled so its `SyntaxError` can be displayed without stopping the notebook. Advanced unpacking syntax is handled later.

In [47]:
invalid_call = 'calculate_cost(item="bananas", quantity=6, 0.74)'
try:
    compile(invalid_call, "<function-call-example>", "exec")
except SyntaxError as error:
    print(type(error).__name__ + ":", error.msg)

SyntaxError: positional argument follows keyword argument


## 3. Returning values and exiting early

Printing and returning are different operations. `print()` displays information. `return` sends an object back to the caller and immediately ends the current function call.

### Returning one value
The caller can store a returned object or use it in a larger expression.

In [ ]:
def identity(x):
    return x

value = identity(42)
print(value)
print(identity(42) + 1)

42
43


### A printed message is not a returned calculation

`calculate_cost()` displays a message but has no explicit return statement. Its returned value is `None`. If a caller needs a result for further calculations, return the number rather than only printing it.

In [ ]:
result = calculate_cost("bananas", 6, 0.74)
print("Returned value:", result)

6 bananas cost $4.44
Returned value: None


### Returning a tuple or another object

Comma-separated return expressions produce one tuple. A function can also return a dictionary or another Python object. The caller can immediately use operations supported by that result.

In [ ]:
def create_point(x, y):
    return x, y

point = create_point(2, 4)
print(point)
x, y = point
print(x, y)

def as_dict():
    return dict(one=1, two=2, three=3)

print(as_dict())
print(as_dict()["two"])

(2, 4)
2 4
{'one': 1, 'two': 2, 'three': 3}
2


### Returning `None`

A bare `return` returns `None`. Reaching the end of a function without a return statement also returns `None`. An explicit `return None` can communicate that a search found no matching result.

In [ ]:
def return_without_value():
    return

print(return_without_value())
print(unfinished_task())

None
None


In [ ]:
def find_user(username, user_list):
    for user in user_list:
        if user["username"] == username:
            return user
    return None

users = [
    {"username": "alice", "email": "alice@example.com"},
    {"username": "bob", "email": "bob@example.com"},
]
print(find_user("alice", users))
print(find_user("linda", users))

{'username': 'alice', 'email': 'alice@example.com'}
None


### Early returns and explicit absence checks

The search ends as soon as a match is found. If no match exists, the final return runs. Check for `None` with `is None` or `is not None` when absence is the condition you intend to test; other false values can have different meanings.

In [ ]:
user = find_user("linda", users)
if user is None:
    print("Linda isn't a registered user")

user = find_user("alice", users)
if user is not None:
    print("Alice's email:", user["email"])

Linda isn't a registered user
Alice's email: alice@example.com


### Predicates return Boolean values

A predicate answers a true-or-false question. Returning the comparison directly is sufficient; an extra `if` that returns `True` or `False` is unnecessary here.

In [ ]:
def is_even(number):
    return number % 2 == 0

print(is_even(2))
print(is_even(3))

True
False


## 4. Scope, mutation, and side effects

Names assigned in a function are normally local to that call. A local name can be separate from an identically named outer variable. However, a parameter and the caller’s variable can refer to the same mutable object.

A *side effect* is an observable interaction beyond simply returning a value, such as printing or changing an existing object. Side effects are not inherently wrong, but a function’s interface should make them clear.

### Mutating an argument
The following function replaces items in the list supplied by the caller.

In [ ]:
def double_in_place(numbers):
    for i, _ in enumerate(numbers):
        numbers[i] *= 2

numbers = [1, 2, 3, 4, 5]
print("Input before the call:", numbers)
returned_value = double_in_place(numbers)
print("Input after the call:", numbers)
print("Returned value:", returned_value)

Input before the call: [1, 2, 3, 4, 5]
Input after the call: [2, 4, 6, 8, 10]
Returned value: None


### Returning a new object instead

Constructing a new list avoids changing this input list. Compare the original list, the returned list, and their identities.

In [ ]:
def double(numbers):
    result = []
    for number in numbers:
        result.append(number * 2)
    return result

numbers = [1, 2, 3, 4, 5]
double_numbers = double(numbers)
print("Returned list:", double_numbers)
print("Original list:", numbers)
print("Same object:", double_numbers is numbers)

Returned list: [2, 4, 6, 8, 10]
Original list: [1, 2, 3, 4, 5]
Same object: False


### Rebinding is different from mutation

Assigning a new object to the local parameter does not rebind the caller’s variable. This differs from replacing items in the shared list.

In [ ]:
def rebind_numbers(numbers):
    numbers = [number * 2 for number in numbers]
    return numbers

numbers = [1, 2, 3, 4, 5]
print("Result:", rebind_numbers(numbers))
print("Caller still holds:", numbers)

Result: [2, 4, 6, 8, 10]
Caller still holds: [1, 2, 3, 4, 5]


### Make shopping-list dependencies explicit

Accepting the dictionary as a parameter allows the same function to display different lists. The `include_quantities` default controls a small formatting choice while the function retains one responsibility: displaying a list.

In [ ]:
def show_list(shopping_list, include_quantities=True):
    for item_name, quantity in shopping_list.items():
        if include_quantities:
            print(f"{quantity}x {item_name}")
        else:
            print(item_name)
    print()

show_list({"Bread": 1, "Milk": 2})

1x Bread
2x Milk



## 5. Default arguments and safe optional inputs

### Required and optional parameters
A parameter with a default can be omitted in a call. An explicitly supplied argument replaces the default for that call. Keyword syntax and optionality are separate ideas: a required argument may be passed by keyword, and an optional argument may be passed positionally.

This version uses a global shopping list so you can focus first on the quantity default. A later version will receive the dictionary explicitly.

In [48]:
default_quantity_list = {}

def add_item_default(item_name, quantity=1):
    if item_name in default_quantity_list:
        default_quantity_list[item_name] += quantity
    else:
        default_quantity_list[item_name] = quantity

add_item_default("Bread")
add_item_default("Milk", 2)
add_item_default(item_name="Bread", quantity=2)
show_list(default_quantity_list)

3x Bread
2x Milk



`item_name` is required. `quantity` defaults to 1. The second Bread addition increases its existing count. The display function already has an optional formatting flag, so it can be used without redefining it.

In [ ]:
show_list(default_quantity_list)
show_list(default_quantity_list, include_quantities=False)

3x Bread
2x Milk

Bread
Milk



### Ordering parameters with defaults

Within the positional parameter sequence, parameters without defaults must precede those with defaults. This rule does not prohibit a required keyword-only parameter after a defaulted parameter; keyword-only syntax is introduced later.

Compile the invalid definition below to examine the syntax error safely.

In [ ]:
invalid_definition = 'def invalid_add_item(shopping_list=None, item_name, quantity):\n    pass\n'
try:
    compile(invalid_definition, "<function-definition-example>", "exec")
except SyntaxError as error:
    print(type(error).__name__ + ":", error.msg)

SyntaxError: parameter without a default follows parameter with a default


### Mutable defaults are shared across calls

Default expressions are evaluated once, when the function is defined; not each time the function is called.

Consequently, this dictionary default is the same object whenever the argument is omitted.


The following example intentionally demonstrates the resulting unwanted sharing.

In [50]:
def add_item_shared(item_name, quantity, shopping_list={}):
    if item_name in shopping_list:
        shopping_list[item_name] += quantity
    else:
        shopping_list[item_name] = quantity
    return shopping_list

clothes_shop_list = add_item_shared("Shirt", 3)
electronics_store_list = add_item_shared("USB cable", 1)

print("Same dictionary:", clothes_shop_list is electronics_store_list)
print('clothes_shop_list',clothes_shop_list)
print('electronics_store_list',electronics_store_list)
show_list(clothes_shop_list)
show_list(electronics_store_list)


Same dictionary: True
clothes_shop_list {'Shirt': 3, 'USB cable': 1}
electronics_store_list {'Shirt': 3, 'USB cable': 1}
3x Shirt
1x USB cable

3x Shirt
1x USB cable



### How can I solve it? Use `None` to request a fresh dictionary

When the caller omits the dictionary, or explicitly passes `None`, create a new dictionary inside the function. If a dictionary is supplied, update that object and return it. This preserves the difference between “no dictionary supplied” and “an empty dictionary supplied.”

In [ ]:
def add_item(item_name, quantity=1, shopping_list=None):
    if shopping_list is None:
        shopping_list = {}
    if item_name in shopping_list:
        shopping_list[item_name] += quantity
    else:
        shopping_list[item_name] = quantity
    return shopping_list

clothes_shop_list = add_item("Shirt", 3)
electronics_store_list = add_item("USB cable")
print("Same dictionary:", clothes_shop_list is electronics_store_list)
print('clothes_shop_list',clothes_shop_list)
print('electronics_store_list',electronics_store_list)
show_list(clothes_shop_list)
show_list(electronics_store_list)

Same dictionary: False
clothes_shop_list {'Shirt': 3}
electronics_store_list {'USB cable': 1}
3x Shirt

1x USB cable



## 6. Variable-length arguments and call unpacking

### Collecting positional arguments with `*`
A parameter preceded by `*` collects the extra positional arguments into a tuple. The name `args` is conventional, not mandatory; a descriptive name such as `item_names` often communicates more.

In [ ]:
def add_items_demo(*item_names):
    print(type(item_names))
    print(item_names)

add_items_demo("Coffee", "Tea", "Cake", "Bread")
add_items_demo()

<class 'tuple'>
('Coffee', 'Tea', 'Cake', 'Bread')
<class 'tuple'>
()


### Using the collected tuple

This function has one required shopping-list argument and any number of additional item names. It assigns quantity 1 to each named item; it does not increment an existing quantity. That behavior differs deliberately from `add_item()`.

In [ ]:
def add_items_by_name(shopping_list, *item_names):
    for item_name in item_names:
        shopping_list[item_name] = 1
    return shopping_list

name_only_list = {}
add_items_by_name(name_only_list, "Coffee", "Tea", "Cake", "Bread")
show_list(name_only_list)



1x Coffee
1x Tea
1x Cake
1x Bread



### Unpacking an iterable at the call site

In a definition, `*item_names` collects arguments. In a call, `*some_items` expands an iterable into positional arguments. Passing the list without `*` instead passes one list object.

In [ ]:
some_items = ["Coffee", "Tea", "Cake", "Bread"]
print(some_items)
print(*some_items)

add_items_demo(some_items)
add_items_demo(*some_items)

['Coffee', 'Tea', 'Cake', 'Bread']
Coffee Tea Cake Bread
<class 'tuple'>
(['Coffee', 'Tea', 'Cake', 'Bread'],)
<class 'tuple'>
('Coffee', 'Tea', 'Cake', 'Bread')


In [ ]:
unpacked_list = {}
add_items_by_name(unpacked_list, *some_items)
show_list(unpacked_list)

1x Coffee
1x Tea
1x Cake
1x Bread



### Collecting keyword arguments with `**`

A parameter preceded by `**kwargs` collects unmatched keyword arguments into a dictionary. The name `kwargs` is conventional. Here, `things_to_buy` makes the meaning clearer.

In [ ]:
def keyword_items_demo(**things_to_buy):
    print(type(things_to_buy))
    print(things_to_buy)

keyword_items_demo(coffee=1, tea=2, cake=1, bread=3)
keyword_items_demo()

<class 'dict'>
{'coffee': 1, 'tea': 2, 'cake': 1, 'bread': 3}
<class 'dict'>
{}


### Using the collected dictionary

The following function sets the quantities supplied through keywords. A later value for an existing key replaces the previous quantity rather than adding to it.

In [ ]:
def add_items_by_quantity(shopping_list, **things_to_buy):
    for item_name, quantity in things_to_buy.items():
        shopping_list[item_name] = quantity
    return shopping_list

quantity_list = {}
add_items_by_quantity(quantity_list, coffee=1, tea=2, cake=1, bread=3)
show_list(quantity_list)

1x coffee
2x tea
1x cake
3x bread



### Unpacking a mapping at the call site

In a call, `**mapping` expands mapping entries into keyword arguments. Keys must be strings. Unless the function collects extra keywords, the keys must match accepted parameter names. Providing a parameter both directly and through unpacking still assigns it twice and raises an error.

In [ ]:
purchase = {"item": "bananas", "quantity": 6, "price": 0.74}
calculate_cost(**purchase)

quantities = {"coffee": 1, "tea": 2, "cake": 1, "bread": 3}
expanded_quantity_list = {}
add_items_by_quantity(expanded_quantity_list, **quantities)
show_list(expanded_quantity_list)

6 bananas cost $4.44
1x coffee
2x tea
1x cake
3x bread



In [ ]:
try:
    calculate_cost("bananas", **purchase)
except TypeError as error:
    print(type(error).__name__ + ":", error)

TypeError: calculate_cost() got multiple values for argument 'item'


## 7. Positional-only and keyword-only parameters

Whether an argument is required is separate from how it must be passed. A signature can restrict a parameter to positional or keyword use.

| Signature feature | Meaning |
|---|---|
| Parameter before `/` | Positional-only |
| Ordinary parameter between `/` and `*` | Positional or keyword |
| Parameter after `*` or `*args` | Keyword-only |
| `*args` | Collect extra positional arguments into a tuple |
| `**kwargs` | Collect extra keyword arguments into a dictionary |
| `=default` | Permit omission of that parameter |

The `/` and bare `*` are separators, not names receiving values.

### Combining the parameter categories

In this example, `x` and `y` are positional-only; `z` and `w` accept either style; `a` and `b` are required keyword-only parameters.

In [ ]:
def demonstrate_signature(x, y, /, z, w, *, a, b):
    print(x, y, z, w, a, b)

demonstrate_signature(1, 2, 3, 4, a="A", b="B")
demonstrate_signature(1, 2, z=3, w=4, a="A", b="B")

1 2 3 4 A B
1 2 3 4 A B


In [ ]:
try:
    demonstrate_signature(1, y=2, z=3, w=4, a="A", b="B")
except TypeError as error:
    print("Positional-only restriction ->", type(error).__name__ + ":", error)

try:
    demonstrate_signature(1, 2, 3, 4, "A", b="B")
except TypeError as error:
    print("Keyword-only restriction ->", type(error).__name__ + ":", error)

Positional-only restriction -> TypeError: demonstrate_signature() got some positional-only arguments passed as keyword arguments: 'y'
Keyword-only restriction -> TypeError: demonstrate_signature() takes 4 positional arguments but 5 positional arguments (and 1 keyword-only argument) were given


### Required keyword-only parameters after a default

A required keyword-only parameter can follow a parameter with a default. This is why “all required parameters must always precede all optional parameters” is too broad. Here, `include_quantities` is optional, while `shopping_list` must be supplied by keyword.

In [51]:
def display_selected_list(include_quantities=True, *, shopping_list):
    show_list(shopping_list, include_quantities=include_quantities)

display_selected_list(shopping_list={"Bread": 1, "Milk": 2})
try:
    display_selected_list()
except TypeError as error:
    print(type(error).__name__ + ":", error)

1x Bread
2x Milk

TypeError: display_selected_list() missing 1 required keyword-only argument: 'shopping_list'


### A keyword-only option after variable positional arguments

Parameters following `*item_names` must be passed by keyword. The collected item names are distinct from the optional quantity setting.

In [ ]:
def add_items_with_default_quantity(shopping_list, *item_names, quantity=1):
    for item_name in item_names:
        add_item(item_name, quantity, shopping_list)
    return shopping_list

bulk_list = {}
add_items_with_default_quantity(bulk_list, "Coffee", "Tea", quantity=2)
show_list(bulk_list)

2x Coffee
2x Tea



## 8. Docstrings and annotations

### Documenting a function
A string literal placed first in a function body becomes its docstring. Explain the task, input expectations, return value, and important side effects. Documenting mutation is especially useful for functions that receive collections.

In [52]:
def double_documented(numbers):
    """Return a new list containing twice each input value.

    Leave the input collection unchanged. Each input value must support
    multiplication by the integer 2.
    """
    return [number * 2 for number in numbers]

print(double_documented.__doc__)
print(double_documented([1, 2, 3]))

Return a new list containing twice each input value.

Leave the input collection unchanged. Each input value must support
multiplication by the integer 2.

[2, 4, 6]


### Type annotations describe expectations

Annotations commonly express expected input and return types. They do not, by themselves, enforce those types at runtime. External type-checking tools can use them to identify inconsistencies.

In [ ]:
def is_even_annotated(number: int) -> bool:
    """Return whether an integer is even."""
    return number % 2 == 0

print(is_even_annotated.__annotations__)
print(is_even_annotated(2))
print(is_even_annotated(2.0))

{'number': <class 'int'>, 'return': <class 'bool'>}
True
True


The float call runs because the operations used in this function also work for that value. This runtime result does not change the declared type contract or imply that every incompatible input will work.

## 9. Generators and closures

These features extend the same definition-and-call model. A generator produces values on demand; a closure preserves access to names from an enclosing function.

### Generator functions
A function containing `yield` is a generator function. Calling it returns a generator iterator. Iteration starts or resumes its body, and each `yield` produces another value without building a list of all results.

In [ ]:
def cumulative_average(numbers):
    total = 0
    for items, number in enumerate(numbers, 1):
        total += number
        yield total / items

values = [5, 3, 8, 2, 5]
for cum_average in cumulative_average(values):
    print(f"Cumulative average: {cum_average:.2f}")

Cumulative average: 5.00
Cumulative average: 4.00
Cumulative average: 5.33
Cumulative average: 4.50
Cumulative average: 4.60


### Iterators are consumed

A generator maintains its progress. Once exhausted, it does not restart automatically. Call the generator function again to obtain a new iterator. Collecting results into a list gives up the benefit of not storing those results.

In [ ]:
averages = cumulative_average(values)
print(next(averages))
print(list(averages))
print(list(averages))

5.0
[4.0, 5.333333333333333, 4.5, 4.6]
[]


### Returning a function object

A function can return another function. The inner function below uses `exponent` from its enclosing function. Each returned function retains access to the enclosing value associated with its creation.

In [53]:
def generate_power(exponent):
    def power(base):
        return base**exponent
    return power

square = generate_power(2)
cube = generate_power(3)
print(square(4))
print(cube(4))

16
64


In [54]:
square

<function __main__.generate_power.<locals>.power(base)>

Notice the difference between `return power` and `return power(...)`: the first returns the function itself; the second would call it and return its result. Here, the caller supplies `base` later when it calls `square()` or `cube()`.

## 10. Putting the pieces together

Use the safe shopping-list functions already defined; do not redefine them. This example combines an explicit dictionary argument, an optional quantity, a variable number of item names, keyword quantities, and optional display formatting.

Read each call before running the cell. Decide whether it adds to an existing quantity or replaces one.

In [ ]:
weekly_list = add_item("Bread")
add_item("Milk", quantity=2, shopping_list=weekly_list)
add_items_with_default_quantity(weekly_list, "Coffee", "Tea")
add_items_by_quantity(weekly_list, Cake=1)
add_item("Bread", 2, weekly_list)

show_list(weekly_list)
show_list(weekly_list, include_quantities=False)

3x Bread
2x Milk
1x Coffee
1x Tea
1x Cake

Bread
Milk
Coffee
Tea
Cake



### Checking behavior with assertions

Assertions make expected behavior explicit. These checks verify the displayed shopping list, a returned value, and the independence of newly created dictionaries. They are small examples of testing, not proof that every possible input has been handled.

In [ ]:
assert weekly_list == {"Bread": 3, "Milk": 2, "Coffee": 1, "Tea": 1, "Cake": 1}
assert double([1, 2, 3]) == [2, 4, 6]
assert find_user("linda", users) is None
first_list = add_item("Shirt", 3)
second_list = add_item("USB cable", 1)
assert first_list is not second_list
assert first_list == {"Shirt": 3}
assert second_list == {"USB cable": 1}
print("All example checks passed.")

All example checks passed.
